# Pet Adoption Classification - part 2

In [18]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier


# sklearn metrics used only to verify our manual implementations
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, roc_curve,
)

colors = ['#9b5de5', '#f15bb5', '#fee440', '#00bbf9', '#00f5d4']

## Data Loading
We loaded 8 CSV files prepared in the previous step

In [3]:
df_simple_std = pd.read_csv('data_after_processing/df_simple_std.csv')
df_simple2_std = pd.read_csv('data_after_processing/df_simple2_std.csv')
df_knn_std = pd.read_csv('data_after_processing/df_knn_std.csv')
df_pycaret_std = pd.read_csv('data_after_processing/df_pycaret_std.csv')

df_simple_minmax = pd.read_csv('data_after_processing/df_simple_minmax.csv')
df_simple2_minmax = pd.read_csv('data_after_processing/df_simple2_minmax.csv')
df_knn_minmax = pd.read_csv('data_after_processing/df_knn_minmax.csv')
df_pycaret_minmax = pd.read_csv('data_after_processing/df_pycaret_minmax.csv')

In [4]:
TARGET = 'AdoptionLikelihood'

For convenient iteration we register all 8 dataset variants in a dictionary.

In [9]:
DATASETS = {
    'simple_std':     df_simple_std,
    'simple2_std':    df_simple2_std,
    'knn_std':        df_knn_std,
    'pycaret_std':    df_pycaret_std,
    'simple_minmax':  df_simple_minmax,
    'simple2_minmax': df_simple2_minmax,
    'knn_minmax':     df_knn_minmax,
    'pycaret_minmax': df_pycaret_minmax,
}

for name, df in DATASETS.items():
    print(f'{name:<16}  shape={df.shape}  '
          f'class balance={df[TARGET].value_counts(normalize=True).round(3).to_dict()}')

simple_std        shape=(2007, 25)  class balance={0: 0.672, 1: 0.328}
simple2_std       shape=(2007, 25)  class balance={0: 0.672, 1: 0.328}
knn_std           shape=(2007, 25)  class balance={0.0: 0.672, 1.0: 0.328}
pycaret_std       shape=(2007, 25)  class balance={0: 0.672, 1: 0.328}
simple_minmax     shape=(2007, 25)  class balance={0: 0.672, 1: 0.328}
simple2_minmax    shape=(2007, 25)  class balance={0: 0.672, 1: 0.328}
knn_minmax        shape=(2007, 25)  class balance={0.0: 0.672, 1.0: 0.328}
pycaret_minmax    shape=(2007, 25)  class balance={0: 0.672, 1: 0.328}


## Manual Train / Test Split

Manual stratified split - without using `train_test_split` from sklearn

In [10]:
def manual_train_test_split(df, target_column, test_size=0.2, random_state=42):
    random_generator = np.random.default_rng(random_state)
    
    train_idx = []
    test_idx = []

    for class_label in df[target_column].unique():
        class_idx = df.index[df[target_column] == class_label].tolist()
        
        random_generator.shuffle(class_idx)
        
        num_test_samples = max(1, int(len(class_idx) * test_size))
        
        test_idx.extend(class_idx[:num_test_samples])
        train_idx.extend(class_idx[num_test_samples:])

    train_set = df.loc[train_idx].sample(frac=1, random_state=random_state).reset_index(drop=True)
    test_set  = df.loc[test_idx].sample(frac=1, random_state=random_state).reset_index(drop=True)

    return train_set, test_set

In [11]:
train_ex, test_ex = manual_train_test_split(df_simple_std, TARGET)

print(f'Train size: {len(train_ex)}, Test size: {len(test_ex)}')
print('Train balance:', train_ex[TARGET].value_counts(normalize=True).round(3).to_dict())
print('Test balance: ', test_ex[TARGET].value_counts(normalize=True).round(3).to_dict())
print('Original:    ', df_simple_std[TARGET].value_counts(normalize=True).round(3).to_dict())

Train size: 1607, Test size: 400
Train balance: {0: 0.671, 1: 0.329}
Test balance:  {0: 0.672, 1: 0.328}
Original:     {0: 0.672, 1: 0.328}


## Manual implementation of classification metrics

For binary classification we implement from scratch:
- **Confusion matrix**
- **Accuracy** = $(TP + TN) / (TP + TN + FP + FN)$
- **Precision** = $TP / (TP + FP)$
- **Recall** = $TP / (TP + FN)$
- **F1-score** = $2 \cdot \frac{Precision \cdot Recall}{Precision + Recall}$
- **ROC-AUC** - via trapezoidal integration over the ROC curve.

In [13]:
def manual_confusion_matrix(y_true, y_pred, num_classes=2):
    '''Build a confusion matrix where rows are true labels and columns are predicted labels.'''
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    cm = np.zeros((num_classes, num_classes), dtype=int)
    for true_label, pred_label in zip(y_true, y_pred):
        cm[true_label, pred_label] += 1
    return cm


def manual_accuracy(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return float(np.mean(y_true == y_pred))


def manual_precision(y_true, y_pred, positive_class=1):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    tp = int(np.sum((y_pred == positive_class) & (y_true == positive_class)))
    fp = int(np.sum((y_pred == positive_class) & (y_true != positive_class)))
    return tp / (tp + fp) if (tp + fp) > 0 else 0.0


def manual_recall(y_true, y_pred, positive_class=1):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    tp = int(np.sum((y_pred == positive_class) & (y_true == positive_class)))
    fn = int(np.sum((y_pred != positive_class) & (y_true == positive_class)))
    return tp / (tp + fn) if (tp + fn) > 0 else 0.0


def manual_f1(y_true, y_pred, positive_class=1):
    p = manual_precision(y_true, y_pred, positive_class)
    r = manual_recall(y_true, y_pred, positive_class)
    return 2 * p * r / (p + r) if (p + r) > 0 else 0.0


def manual_roc_auc(y_true, y_score):
    '''Compute ROC-AUC via trapezoidal integration of the ROC curve.'''
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)

    # Sort samples by descending score
    order = np.argsort(-y_score, kind='mergesort')
    y_true_sorted = y_true[order]

    P = int(np.sum(y_true == 1))
    N = int(np.sum(y_true == 0))
    if P == 0 or N == 0:
        return 0.5

    # TPR = cumulative positives / total positives
    # FPR = cumulative negatives / total negatives
    tpr = np.cumsum(y_true_sorted == 1) / P
    fpr = np.cumsum(y_true_sorted == 0) / N

    # Prepend (0, 0) so curve starts at origin
    tpr = np.concatenate(([0.0], tpr))
    fpr = np.concatenate(([0.0], fpr))

    return float(np.trapezoid(tpr, fpr))

### Verification against sklearn

In [20]:
rng_check = np.random.default_rng(0)
y_true_chk = rng_check.integers(0, 2, size=500)
y_score_chk = 0.3 * y_true_chk + rng_check.random(500)
y_score_chk = (y_score_chk - y_score_chk.min()) / (y_score_chk.max() - y_score_chk.min())
y_pred_chk = (y_score_chk > 0.5).astype(int)

print(f'{'metric':<12} {'manual':>10} {'sklearn':>10}')
print('-' * 34)
print(f'{'accuracy':<12} {manual_accuracy(y_true_chk, y_pred_chk):>10.6f} {accuracy_score(y_true_chk, y_pred_chk):>10.6f}')
print(f'{'precision':<12} {manual_precision(y_true_chk, y_pred_chk):>10.6f} {precision_score(y_true_chk, y_pred_chk):>10.6f}')
print(f'{'recall':<12} {manual_recall(y_true_chk, y_pred_chk):>10.6f} {recall_score(y_true_chk, y_pred_chk):>10.6f}')
print(f'{'f1':<12} {manual_f1(y_true_chk, y_pred_chk):>10.6f} {f1_score(y_true_chk, y_pred_chk):>10.6f}')
print(f'{'roc_auc':<12} {manual_roc_auc(y_true_chk, y_score_chk):>10.6f} {roc_auc_score(y_true_chk, y_score_chk):>10.6f}')

print('\nConfusion matrix (manual):')
print(manual_confusion_matrix(y_true_chk, y_pred_chk))
print('Confusion matrix (sklearn):')
print(confusion_matrix(y_true_chk, y_pred_chk))

metric           manual    sklearn
----------------------------------
accuracy       0.670000   0.670000
precision      0.713178   0.713178
recall         0.669091   0.669091
f1             0.690432   0.690432
roc_auc        0.776178   0.776178

Confusion matrix (manual):
[[151  74]
 [ 91 184]]
Confusion matrix (sklearn):
[[151  74]
 [ 91 184]]


## Manual Stratified Cross-Validation

In [19]:
def manual_stratified_kfold(df, target_column, n_splits=5, random_state=42):
    '''Generator yielding stratified (train_idx, val_idx) splits as positional indices.'''
    rng = np.random.default_rng(random_state)

    folds_per_class = {}
    for class_label in sorted(df[target_column].unique()):
        positions = np.where(df[target_column].to_numpy() == class_label)[0]
        rng.shuffle(positions)
        # Distribute this class's samples evenly into n_splits folds
        folds_per_class[class_label] = np.array_split(positions, n_splits)

    for fold_i in range(n_splits):
        val_idx = np.concatenate([folds_per_class[c][fold_i] for c in folds_per_class])
        train_idx = np.concatenate([
            folds_per_class[c][j]
            for c in folds_per_class
            for j in range(n_splits) if j != fold_i
        ])

        rng.shuffle(train_idx)
        rng.shuffle(val_idx)

        yield train_idx, val_idx

In [21]:
print(f'Original balance: {df_simple_std[TARGET].value_counts(normalize=True).round(3).to_dict()}\n')

for fold_i, (tr_idx, vl_idx) in enumerate(manual_stratified_kfold(df_simple_std, TARGET, n_splits=5)):
    tr_bal = df_simple_std.iloc[tr_idx][TARGET].value_counts(normalize=True).round(3).to_dict()
    vl_bal = df_simple_std.iloc[vl_idx][TARGET].value_counts(normalize=True).round(3).to_dict()
    overlap = len(set(tr_idx) & set(vl_idx))
    print(f'Fold {fold_i + 1}: train={len(tr_idx)} {tr_bal}  val={len(vl_idx)} {vl_bal}  overlap={overlap}')

Original balance: {0: 0.672, 1: 0.328}

Fold 1: train=1605 {0: 0.672, 1: 0.328}  val=402 {0: 0.672, 1: 0.328}  overlap=0
Fold 2: train=1605 {0: 0.672, 1: 0.328}  val=402 {0: 0.672, 1: 0.328}  overlap=0
Fold 3: train=1605 {0: 0.672, 1: 0.328}  val=402 {0: 0.672, 1: 0.328}  overlap=0
Fold 4: train=1606 {0: 0.672, 1: 0.328}  val=401 {0: 0.671, 1: 0.329}  overlap=0
Fold 5: train=1607 {0: 0.671, 1: 0.329}  val=400 {0: 0.672, 1: 0.328}  overlap=0
